# Consumer Complaint Classification: SimpleRNN, LSTM, GRU
This notebook trains three sequence models from scratch with Keras and compares them.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Bidirectional, Dropout
from tensorflow.keras.callbacks import EarlyStopping
print("Imports done")

## 2. Load and explore the data
The dataset used here is already preprocessed (lowercase, no punctuation, numbers or stop words, lemmatized).

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/shashwatwork/consume-complaints-dataset-fo-nlp/complaints_processed.csv")
df = df.drop(columns=["Unnamed: 0"])
df.columns = ["label", "text"]
df = df.dropna()
print(df.shape)

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(y=df["label"], order=df["label"].value_counts().index)
plt.title("Class Distribution")
plt.show()
print(df["label"].value_counts())

## 3. Balance the classes
The classes are very imbalanced, so we take the same number of rows from each class.

In [ ]:
# keep 12000 rows from each class so the classes are balanced
df = df.groupby("label").sample(n=12000, random_state=42)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(df["label"].value_counts())

## 4. Train, validation and test split

In [ ]:
# split the data into 70% train, 15% validation, 15% test
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)
print(train_df.shape, val_df.shape, test_df.shape)

In [ ]:
# turn the class names into numbers 0 to 4
encoder = LabelEncoder()
y_train = encoder.fit_transform(train_df["label"])
y_val = encoder.transform(val_df["label"])
y_test = encoder.transform(test_df["label"])
print(encoder.classes_)

## 5. Tokenizer and padding

In [ ]:
# the tokenizer learns the words from the train set only
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_df["text"]), maxlen=200)
X_val = pad_sequences(tokenizer.texts_to_sequences(val_df["text"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_df["text"]), maxlen=200)
print(X_train.shape)

In [ ]:
results = {}
print(results)

## SimpleRNN

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
rnn_model = Sequential([Embedding(20000, 128), Bidirectional(SimpleRNN(64)), Dropout(0.5), Dense(5, activation="softmax")])
rnn_model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history_rnn_model = rnn_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=64, callbacks=[early_stop])
print("Training finished")

In [ ]:
# argmax picks the class with the highest probability
y_pred = np.argmax(rnn_model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred, target_names=encoder.classes_))
results["SimpleRNN"] = [accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average="macro"), recall_score(y_test, y_pred, average="macro"), f1_score(y_test, y_pred, average="macro")]
print(results)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title("SimpleRNN Confusion Matrix")
plt.show()
print(cm)

## LSTM

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
lstm_model = Sequential([Embedding(20000, 128), Bidirectional(LSTM(64)), Dropout(0.5), Dense(5, activation="softmax")])
lstm_model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history_lstm_model = lstm_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=64, callbacks=[early_stop])
print("Training finished")

In [ ]:
# argmax picks the class with the highest probability
y_pred = np.argmax(lstm_model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred, target_names=encoder.classes_))
results["LSTM"] = [accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average="macro"), recall_score(y_test, y_pred, average="macro"), f1_score(y_test, y_pred, average="macro")]
print(results)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title("LSTM Confusion Matrix")
plt.show()
print(cm)

## GRU

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
gru_model = Sequential([Embedding(20000, 128), Bidirectional(GRU(64)), Dropout(0.5), Dense(5, activation="softmax")])
gru_model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history_gru_model = gru_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=64, callbacks=[early_stop])
print("Training finished")

In [ ]:
# argmax picks the class with the highest probability
y_pred = np.argmax(gru_model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred, target_names=encoder.classes_))
results["GRU"] = [accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average="macro"), recall_score(y_test, y_pred, average="macro"), f1_score(y_test, y_pred, average="macro")]
print(results)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title("GRU Confusion Matrix")
plt.show()
print(cm)

## Save the results

In [ ]:
json.dump(results, open("results.json", "w"))
print(results)